# PriceFusion V0.4.1 — Structured Parser Hardening

**Project:** PriceFusion — Multimodal Product Price Intelligence  
**Stage:** V0.4.1  
**Goal:** Harden the structured parser from V0.4 before feeding engineered features into V0.5 models.

This patch focuses on:
- stricter pack/case/set/count extraction
- safer quantity caps
- better unit normalization
- removal of invalid/placeholder units
- safer quantity multiplier construction
- before/after diagnostics for suspicious large quantities
- regenerated V0.4 feature parquet files for V0.5


## 1. Imports

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 220)

print('V0.4.1 environment ready.')

V0.4.1 environment ready.


## 2. Resolve Paths and Load Data

In [2]:
cwd = Path.cwd().resolve()

if (cwd / 'dataset').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'dataset').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Could not find dataset/.')

DATASET_DIR = PROJECT_ROOT / 'dataset'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FEATURES_DIR = ARTIFACTS_DIR / 'features'
SPLITS_DIR = ARTIFACTS_DIR / 'splits'

TRAIN_CSV = DATASET_DIR / 'train.csv'
TEST_CSV = DATASET_DIR / 'test.csv'
TRAIN_IDS_PATH = SPLITS_DIR / 'train_ids.csv'
VALID_IDS_PATH = SPLITS_DIR / 'validation_ids.csv'

train_raw = pd.read_csv(TRAIN_CSV)
test_raw = pd.read_csv(TEST_CSV)
train_ids = pd.read_csv(TRAIN_IDS_PATH)['sample_id']
valid_ids = pd.read_csv(VALID_IDS_PATH)['sample_id']

print('Train raw:', train_raw.shape)
print('Test raw :', test_raw.shape)

Train raw: (75000, 4)
Test raw : (75000, 3)


## 3. Helpers

In [3]:
def safe_text(x):
    return '' if pd.isna(x) else str(x)

def extract_line_value(text, label):
    pattern = rf'(?im)^{re.escape(label)}\s*:\s*([^\n\r]+)'
    m = re.search(pattern, safe_text(text))
    return m.group(1).strip() if m else None

def parse_float(value):
    if value is None:
        return np.nan
    value = str(value).strip().replace(',', '')
    if value.lower() in {'', 'nan', 'none', 'null', 'n/a', 'na', '---'}:
        return np.nan
    try:
        return float(value)
    except ValueError:
        return np.nan

def first_valid_int(patterns, text, min_value=1, max_value=None):
    text = safe_text(text)
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if not m:
            continue
        try:
            val = int(m.group(1).replace(',', ''))
        except Exception:
            continue
        if val < min_value:
            continue
        if max_value is not None and val > max_value:
            continue
        return val
    return np.nan


## 4. Hardened Unit Normalization

In [4]:
INVALID_UNITS = {
    '', 'none', 'nan', 'null', 'n/a', 'na', '---', '-', 'unknown', '1',
    'product_weight'
}

UNIT_MAP = {
    'oz': 'ounce', 'ounce': 'ounce', 'ounces': 'ounce',
    'fl oz': 'fl_oz', 'fl. oz': 'fl_oz', 'fl. oz.': 'fl_oz',
    'fl.oz': 'fl_oz', 'fl ounce': 'fl_oz',
    'fluid ounce': 'fl_oz', 'fluid ounces': 'fl_oz', 'fluid ounce(s)': 'fl_oz',
    'lb': 'pound', 'lbs': 'pound', 'pound': 'pound', 'pounds': 'pound',
    'g': 'gram', 'gr': 'gram', 'gram': 'gram', 'grams': 'gram', 'gramm': 'gram',
    'kg': 'kilogram', 'kilogram': 'kilogram', 'kilograms': 'kilogram',
    'ml': 'milliliter', 'milliliter': 'milliliter', 'milliliters': 'milliliter',
    'millilitre': 'milliliter', 'millilitres': 'milliliter',
    'l': 'liter', 'ltr': 'liter', 'liter': 'liter', 'liters': 'liter',
    'litre': 'liter', 'litres': 'liter',
    'count': 'count', 'counts': 'count', 'ct': 'count',
    'each': 'count', 'piece': 'count', 'pieces': 'count',
    'pack': 'count', 'packs': 'count', 'bottle': 'count', 'bottles': 'count',
    'can': 'count', 'cans': 'count', 'bag': 'count', 'bags': 'count',
    'carton': 'count', 'per carton': 'count', 'per box': 'count', 'box': 'count',
    'per package': 'count', 'pouch': 'count', 'tea bags': 'count',
    'ziplock bags': 'count',
}

def normalize_unit(unit):
    if unit is None or pd.isna(unit):
        return 'unknown'
    u = re.sub(r'\s+', ' ', str(unit).strip().lower())
    if u in INVALID_UNITS:
        return 'unknown'
    return UNIT_MAP.get(u, u)


## 5. Hardened Quantity Parsers

In [5]:
# Strong-context patterns only. These intentionally avoid generic number matching.
PACK_PATTERNS = [
    r'\bpack\s+of\s+(\d{1,3})\b',
    r'\b(\d{1,3})\s*[- ]pack\b',
    r'\b(\d{1,3})\s+pack\b',
    r'\bpk[- ]?(\d{1,3})\b',
    r'\bpack[- ]?(\d{1,3})\b',
]

CASE_PATTERNS = [
    r'\b(\d{1,3})\s+per\s+case\b',
    r'\bcase\s+of\s+(\d{1,3})\b',
    r'\b(\d{1,3})\s*[- ]case\b',
]

SET_PATTERNS = [
    r'\bset\s+of\s+(\d{1,3})\b',
    r'\b(\d{1,3})\s*[- ]piece\s+set\b',
]

SERVING_PATTERNS = [
    r'\b(\d{1,5})\s+servings?\b',
]

PALLET_PATTERNS = [
    r'\bpallet\s+of\s+(\d{1,3})\s+cases?\b',
    r'\b(\d{1,3})\s+cases?\s+(?:on|per)\s+pallet\b',
]

COUNT_PATTERNS = [
    r'\b(\d{1,5})\s*[- ]count\b',
    r'\b(\d{1,5})\s*ct\b',
    r'\b(\d{1,5})\s*pcs?\b',
    r'\b(\d{1,5})\s*pieces?\b',
    r'\b(\d{1,5})\s*bottles?\b',
    r'\b(\d{1,5})\s*cans?\b',
    r'\b(\d{1,5})\s*bags?\b',
    r'\b(\d{1,5})\s*packets?\b',
]

def extract_pack_quantity(text):
    v = first_valid_int(PACK_PATTERNS, text, 1, 500)
    return int(v) if not np.isnan(v) else 1

def extract_case_quantity(text):
    v = first_valid_int(CASE_PATTERNS, text, 1, 500)
    return int(v) if not np.isnan(v) else 1

def extract_set_quantity(text):
    v = first_valid_int(SET_PATTERNS, text, 1, 200)
    return int(v) if not np.isnan(v) else 1

def extract_servings(text):
    return first_valid_int(SERVING_PATTERNS, safe_text(text).replace(',', ''), 1, 50000)

def extract_pallet_cases(text):
    return first_valid_int(PALLET_PATTERNS, text, 1, 500)

def extract_explicit_count(text):
    return first_valid_int(COUNT_PATTERNS, text, 1, 50000)

## 6. Measurement Parsers

In [6]:
MEASUREMENT_PATTERNS = {
    'ounce': [r'\b(\d+(?:\.\d+)?)\s*(?:ounce|ounces|oz)\b'],
    'fl_oz': [r'\b(\d+(?:\.\d+)?)\s*(?:fl\.?\s*oz\.?|fluid\s+ounces?|fluid\s+ounce)\b'],
    'pound': [r'\b(\d+(?:\.\d+)?)\s*(?:pounds?|lbs?|lb)\b'],
    'gram': [r'\b(\d+(?:\.\d+)?)\s*(?:grams?|gramm|g)\b'],
    'kilogram': [r'\b(\d+(?:\.\d+)?)\s*(?:kilograms?|kg)\b'],
    'milliliter': [r'\b(\d+(?:\.\d+)?)\s*(?:milliliters?|millilitres?|ml)\b'],
    'liter': [r'\b(\d+(?:\.\d+)?)\s*(?:liters?|litres?|l)\b'],
}

def first_float(patterns, text, max_value=1_000_000):
    text = safe_text(text)
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if not m:
            continue
        try:
            val = float(m.group(1))
        except Exception:
            continue
        if 0 <= val <= max_value:
            return val
    return np.nan

def oz_to_grams(x): return x * 28.349523125
def lb_to_grams(x): return x * 453.59237
def kg_to_grams(x): return x * 1000.0
def fl_oz_to_ml(x): return x * 29.5735295625
def liter_to_ml(x): return x * 1000.0


## 7. Hardened Feature Engineering Function

In [7]:
def engineer_features_hardened(df):
    out = pd.DataFrame(index=df.index)
    out['sample_id'] = df['sample_id'].values
    text = df['catalog_content'].fillna('').astype(str)

    out['item_name'] = text.map(lambda x: extract_line_value(x, 'Item Name'))
    out['raw_value'] = text.map(lambda x: parse_float(extract_line_value(x, 'Value')))
    out['raw_unit'] = text.map(lambda x: extract_line_value(x, 'Unit'))
    out['unit_normalized'] = out['raw_unit'].map(normalize_unit)

    out['catalog_char_count'] = text.str.len()
    out['catalog_word_count'] = text.str.split().str.len()
    out['catalog_line_count'] = text.str.count(r'\n') + 1
    out['catalog_digit_count'] = text.str.count(r'\d')
    out['numeric_token_count'] = text.str.count(r'\b\d+(?:\.\d+)?\b')
    out['bullet_count'] = text.str.count(r'(?i)Bullet Point')
    out['uppercase_count'] = text.str.count(r'\b[A-Z]{2,}\b')
    out['special_char_count'] = text.str.count(r'[^A-Za-z0-9\s]')

    out['pack_quantity'] = text.map(extract_pack_quantity)
    out['case_quantity'] = text.map(extract_case_quantity)
    out['set_quantity'] = text.map(extract_set_quantity)
    out['servings'] = text.map(extract_servings)
    out['pallet_cases'] = text.map(extract_pallet_cases)
    out['explicit_count'] = text.map(extract_explicit_count)

    lower = text.str.lower()
    out['has_pack'] = lower.str.contains(r'\bpack\b', regex=True).astype('int8')
    out['has_case'] = lower.str.contains(r'\bcase\b', regex=True).astype('int8')
    out['has_pallet'] = lower.str.contains(r'\bpallet\b', regex=True).astype('int8')
    out['has_bulk'] = lower.str.contains(r'\bbulk\b', regex=True).astype('int8')
    out['has_serving'] = lower.str.contains(r'\bservings?\b', regex=True).astype('int8')
    out['has_bottle'] = lower.str.contains(r'\bbottles?\b', regex=True).astype('int8')
    out['has_can'] = lower.str.contains(r'\bcans?\b', regex=True).astype('int8')
    out['has_bag'] = lower.str.contains(r'\bbags?\b', regex=True).astype('int8')
    out['has_frozen'] = lower.str.contains(r'\bfrozen\b', regex=True).astype('int8')
    out['has_organic'] = lower.str.contains(r'\borganic\b', regex=True).astype('int8')

    out['measurement_ounce'] = text.map(lambda x: first_float(MEASUREMENT_PATTERNS['ounce'], x))
    out['measurement_fl_oz'] = text.map(lambda x: first_float(MEASUREMENT_PATTERNS['fl_oz'], x))
    out['measurement_pound'] = text.map(lambda x: first_float(MEASUREMENT_PATTERNS['pound'], x))
    out['measurement_gram'] = text.map(lambda x: first_float(MEASUREMENT_PATTERNS['gram'], x))
    out['measurement_kilogram'] = text.map(lambda x: first_float(MEASUREMENT_PATTERNS['kilogram'], x))
    out['measurement_ml'] = text.map(lambda x: first_float(MEASUREMENT_PATTERNS['milliliter'], x))
    out['measurement_liter'] = text.map(lambda x: first_float(MEASUREMENT_PATTERNS['liter'], x))

    out['raw_weight_grams'] = np.nan
    out['raw_volume_ml'] = np.nan

    for unit, fn in [('ounce', oz_to_grams), ('pound', lb_to_grams), ('kilogram', kg_to_grams)]:
        mask = out['unit_normalized'].eq(unit)
        out.loc[mask, 'raw_weight_grams'] = out.loc[mask, 'raw_value'].map(fn)

    mask = out['unit_normalized'].eq('gram')
    out.loc[mask, 'raw_weight_grams'] = out.loc[mask, 'raw_value']

    mask = out['unit_normalized'].eq('fl_oz')
    out.loc[mask, 'raw_volume_ml'] = out.loc[mask, 'raw_value'].map(fl_oz_to_ml)
    mask = out['unit_normalized'].eq('milliliter')
    out.loc[mask, 'raw_volume_ml'] = out.loc[mask, 'raw_value']
    mask = out['unit_normalized'].eq('liter')
    out.loc[mask, 'raw_volume_ml'] = out.loc[mask, 'raw_value'].map(liter_to_ml)

    weight_candidates = pd.DataFrame({
        'oz': out['measurement_ounce'].map(oz_to_grams),
        'lb': out['measurement_pound'].map(lb_to_grams),
        'g': out['measurement_gram'],
        'kg': out['measurement_kilogram'].map(kg_to_grams),
    })
    out['parsed_weight_grams'] = weight_candidates.bfill(axis=1).iloc[:, 0]

    volume_candidates = pd.DataFrame({
        'fl_oz': out['measurement_fl_oz'].map(fl_oz_to_ml),
        'ml': out['measurement_ml'],
        'liter': out['measurement_liter'].map(liter_to_ml),
    })
    out['parsed_volume_ml'] = volume_candidates.bfill(axis=1).iloc[:, 0]

    # Conservative multiplier: avoid multiplying two independently parsed packaging counts.
    out['quantity_multiplier'] = pd.concat([
        out['pack_quantity'],
        out['case_quantity'],
        out['set_quantity'],
    ], axis=1).max(axis=1)
    out['quantity_multiplier'] = out['quantity_multiplier'].clip(lower=1, upper=500)

    out['total_parsed_weight_grams'] = out['parsed_weight_grams'] * out['quantity_multiplier']
    out['total_parsed_volume_ml'] = out['parsed_volume_ml'] * out['quantity_multiplier']

    out['effective_count'] = pd.concat([
        out['explicit_count'],
        out['servings'],
        out['pallet_cases'],
        out['quantity_multiplier'],
    ], axis=1).max(axis=1, skipna=True)

    out['effective_count'] = out['effective_count'].clip(lower=1, upper=50000)

    log_cols = [
        'raw_value', 'catalog_char_count', 'catalog_word_count',
        'numeric_token_count', 'pack_quantity', 'case_quantity',
        'servings', 'pallet_cases', 'explicit_count',
        'raw_weight_grams', 'raw_volume_ml',
        'parsed_weight_grams', 'parsed_volume_ml',
        'total_parsed_weight_grams', 'total_parsed_volume_ml',
        'effective_count'
    ]

    for col in log_cols:
        out[f'log1p_{col}'] = np.log1p(out[col].clip(lower=0))

    return out

## 8. Generate Hardened Features

In [8]:
train_all = engineer_features_hardened(train_raw)
test_all = engineer_features_hardened(test_raw)
train_all['price'] = train_raw['price'].values

print('Train features:', train_all.shape)
print('Test features :', test_all.shape)

Train features: (75000, 61)
Test features : (75000, 60)


## 9. Quantity Sanity Checks

In [9]:
quantity_cols = [
    'pack_quantity', 'case_quantity', 'set_quantity',
    'servings', 'pallet_cases', 'explicit_count',
    'quantity_multiplier', 'effective_count'
]

display(train_all[quantity_cols].describe(percentiles=[0.95, 0.99, 0.999]).T)

,count,mean,std,min,50%,95%,99%,99.9%,max
pack_quantity,75000.0,3.834640,11.349876,1.0,1.0,12.0,32.00,160.002,500.0
case_quantity,75000.0,1.705693,10.198964,1.0,1.0,1.0,12.00,180.000,500.0
set_quantity,75000.0,1.058973,1.707228,1.0,1.0,1.0,1.00,15.001,150.0
servings,1610.0,105.142236,1295.017138,1.0,15.0,200.0,1791.00,6955.000,50000.0
pallet_cases,2.0,81.000000,4.242641,78.0,81.0,83.7,83.94,83.994,84.0
explicit_count,10797.0,58.032880,466.258514,1.0,16.0,150.0,750.00,2522.400,35180.0
quantity_multiplier,75000.0,4.534147,14.888611,1.0,1.0,12.0,48.00,200.004,500.0
effective_count,75000.0,14.575387,260.764689,1.0,1.0,40.0,160.00,1000.000,50000.0


In [10]:
print('Rows with quantity_multiplier > 500:', (train_all['quantity_multiplier'] > 500).sum())
print('Rows with effective_count > 50,000:', (train_all['effective_count'] > 50000).sum())

display(
    train_all.nlargest(25, 'effective_count')[
        ['sample_id', 'price', 'pack_quantity', 'case_quantity', 'set_quantity',
         'servings', 'pallet_cases', 'explicit_count', 'quantity_multiplier', 'effective_count']
    ]
)

Rows with quantity_multiplier > 500: 0
Rows with effective_count > 50,000: 0


,sample_id,price,pack_quantity,case_quantity,set_quantity,servings,pallet_cases,explicit_count,quantity_multiplier,effective_count
50136,23683,28.840,1,1,1,50000.0,NaN,NaN,1,50000.0
51451,154377,35.390,1,1,1,NaN,NaN,35180.0,1,35180.0
17229,23234,21.500,1,3,1,NaN,NaN,24514.0,3,24514.0
5397,288279,63.040,1,1,1,10000.0,NaN,NaN,1,10000.0
67472,216859,95.290,1,1,1,NaN,NaN,9000.0,1,9000.0
1805,298632,37.630,1,1,1,NaN,NaN,6802.0,1,6802.0
14519,255824,61.950,1,1,1,NaN,NaN,6523.0,1,6523.0
33460,255820,64.240,1,1,1,NaN,NaN,6082.0,1,6082.0
67862,55445,93.510,4,1,1,NaN,NaN,6000.0,4,6000.0
5322,210632,26.990,1,1,1,5000.0,NaN,NaN,1,5000.0


## 10. Check Unit Cleanup

In [11]:
unit_summary = (
    train_all['unit_normalized']
    .value_counts(dropna=False)
    .rename_axis('unit_normalized')
    .reset_index(name='count')
)
unit_summary['pct'] = unit_summary['count'] / len(train_all) * 100
display(unit_summary.head(30))

print('Unknown unit rows:', (train_all['unit_normalized'] == 'unknown').sum())

,unit_normalized,count,pct
0,ounce,44028,58.704000
1,count,18345,24.460000
2,fl_oz,11338,15.117333
3,unknown,947,1.262667
4,pound,240,0.320000
5,gram,50,0.066667
6,milliliter,14,0.018667
7,liter,8,0.010667
8,kilogram,6,0.008000
9,jar,4,0.005333


Unknown unit rows: 947


## 11. Align with Permanent V0.3 Split

In [12]:
train_features = train_all[train_all['sample_id'].isin(set(train_ids))].copy()
valid_features = train_all[train_all['sample_id'].isin(set(valid_ids))].copy()

assert len(train_features) == len(train_ids)
assert len(valid_features) == len(valid_ids)

print('Train rows:', len(train_features))
print('Valid rows:', len(valid_features))

Train rows: 60000
Valid rows: 15000


## 12. Build Modeling Tables

In [13]:
DROP_TEXT_COLUMNS = ['item_name', 'raw_unit']

def modeling_table(df, include_price=False):
    cols = [c for c in df.columns if c not in DROP_TEXT_COLUMNS]
    if not include_price:
        cols = [c for c in cols if c != 'price']
    return df[cols].copy()

train_model = modeling_table(train_features, include_price=True)
valid_model = modeling_table(valid_features, include_price=True)
test_model = modeling_table(test_all, include_price=False)

print('Train model table:', train_model.shape)
print('Valid model table:', valid_model.shape)
print('Test model table :', test_model.shape)

Train model table: (60000, 59)
Valid model table: (15000, 59)
Test model table : (75000, 58)


## 13. Save Hardened Artifacts

In [14]:
# Overwrite the V0.4 feature files intentionally so V0.5 uses the hardened parser.
train_model.to_parquet(FEATURES_DIR / 'v04_train_features.parquet', index=False)
valid_model.to_parquet(FEATURES_DIR / 'v04_validation_features.parquet', index=False)
test_model.to_parquet(FEATURES_DIR / 'v04_test_features.parquet', index=False)

unit_summary.to_csv(FEATURES_DIR / 'v041_unit_summary.csv', index=False)
train_model.head(1000).to_csv(FEATURES_DIR / 'v041_train_features_sample.csv', index=False)

print('Hardened parquet artifacts saved.')

Hardened parquet artifacts saved.


## 14. Save Patch Manifest

In [15]:
patch_manifest = {
    'version': 'V0.4.1',
    'base_version': 'V0.4',
    'train_rows': int(len(train_model)),
    'validation_rows': int(len(valid_model)),
    'test_rows': int(len(test_model)),
    'feature_count': int(len([c for c in train_model.columns if c != 'price'])),
    'quantity_multiplier_cap': 500,
    'effective_count_cap': 50000,
    'unit_unknown_rows': int((train_all['unit_normalized'] == 'unknown').sum()),
    'max_quantity_multiplier': float(train_all['quantity_multiplier'].max()),
    'max_effective_count': float(train_all['effective_count'].max()),
    'notes': [
        'Hardened pack/case/set patterns',
        'Removed invalid unit labels',
        'Used max packaging quantity instead of multiplying pack*case*set',
        'Regenerated V0.4 parquet files for V0.5'
    ]
}

with (FEATURES_DIR / 'v041_patch_manifest.json').open('w') as f:
    json.dump(patch_manifest, f, indent=2)

display(pd.Series(patch_manifest, name='value').to_frame())

,value
version,V0.4.1
base_version,V0.4
train_rows,60000
validation_rows,15000
test_rows,75000
feature_count,58
quantity_multiplier_cap,500
effective_count_cap,50000
unit_unknown_rows,947
max_quantity_multiplier,500.0


## 15. Final V0.4.1 Summary

In [16]:
print('=' * 82)
print('PRICEFUSION V0.4.1 — PARSER HARDENING COMPLETE')
print('=' * 82)
print(f'Train rows               : {len(train_model):,}')
print(f'Validation rows          : {len(valid_model):,}')
print(f'Test rows                : {len(test_model):,}')
print(f'Feature count            : {patch_manifest["feature_count"]:,}')
print(f'Max quantity multiplier  : {train_all["quantity_multiplier"].max():,.0f}')
print(f'Max effective count      : {train_all["effective_count"].max():,.0f}')
print(f'Unknown unit rows        : {(train_all["unit_normalized"] == "unknown").sum():,}')
print('=' * 82)
print('V0.4 artifacts are now hardened and ready for V0.5 modeling.')
print('Next milestone: PriceFusion V0.5 — CatBoost / LightGBM Structured Modeling')

PRICEFUSION V0.4.1 — PARSER HARDENING COMPLETE
Train rows               : 60,000
Validation rows          : 15,000
Test rows                : 75,000
Feature count            : 58
Max quantity multiplier  : 500
Max effective count      : 50,000
Unknown unit rows        : 947
V0.4 artifacts are now hardened and ready for V0.5 modeling.
Next milestone: PriceFusion V0.5 — CatBoost / LightGBM Structured Modeling
